In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
CANDIDATES_PATH = Path('/home/msp25gd/ResearchProjectMSc/HR/results/QuickSearch_V2/candidates_-3.5sig_1.5cut_2width_V2.npy')
FULL_META_PATH = Path('/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl')
OUT_DIR = Path('/home/msp25gd/ResearchProjectMSc/ResolutionHandling')

candidates = np.load(CANDIDATES_PATH, allow_pickle=True)
cand_set = set(map(str, candidates.tolist()))

full_meta = pd.read_pickle(FULL_META_PATH)
name_col = 'Reduced' if 'Reduced' in full_meta.columns else 'Object'

filtered = full_meta[full_meta[name_col].astype(str).isin(cand_set)][[name_col, 'SPEC_RES']].copy()
group_nunique = filtered.groupby(name_col)['SPEC_RES'].nunique(dropna=True)

uniform_groups = group_nunique[group_nunique == 1].index.to_numpy(dtype=object)
mixed_groups = group_nunique[group_nunique > 1].index.to_numpy(dtype=object)

np.save(OUT_DIR / 'uniform_groups.npy', uniform_groups)
np.save(OUT_DIR / 'mixed_groups.npy', mixed_groups)
np.save(OUT_DIR / 'all_candidate_groups.npy', np.array(sorted(cand_set), dtype=object))

summary = pd.DataFrame({
    'category': ['total_candidates', 'uniform_groups', 'mixed_groups'],
    'count': [len(cand_set), len(uniform_groups), len(mixed_groups)]
})
summary['percentage'] = 100.0 * summary['count'] / summary.loc[0, 'count']
summary.to_csv(OUT_DIR / 'logs' / 'group_resolution_summary.csv', index=False)

print(summary.to_string(index=False))
print('\nSaved: uniform_groups.npy, mixed_groups.npy, all_candidate_groups.npy')

        category  count  percentage
total_candidates    498  100.000000
  uniform_groups    333   66.867470
    mixed_groups    164   32.931727

Saved: uniform_groups.npy, mixed_groups.npy, all_candidate_groups.npy
